# Spark MLlib – Preprocesamiento de Datos

**Curso:** Computación de Alto Desempeño – MLlib Spark  
**Autor:** Santiago Gil Gallego (Sgg)  
**Fecha:** 27 de noviembre de 2025  

**Objetivo del cuaderno:**  
Este cuaderno realiza el preprocesamiento de dos conjuntos de datos (supervisado y no supervisado) usando Apache Spark MLlib sobre el clúster de la asignatura. Se construyen vistas minables en formato Parquet que serán utilizadas en los cuadernos de técnicas supervisadas y no supervisadas.


En este cuaderno se realiza el preprocesamiento de dos conjuntos de datos:

1. Dataset supervisado: Breast Cancer (`breast_cancer_sgg.csv`).
2. Dataset no supervisado: Iris (`iris_sgg.csv`).


In [1]:
import findspark
findspark.init()

from pyspark import SparkConf
from pyspark.sql import SparkSession, SQLContext

configuraSgg = (
    SparkConf()
    .set("spark.scheduler.mode", "FAIR")
    .set("spark.executor.cores", "1")
    .set("spark.executor.memory", "4G")   #
    .set("spark.cores.max", "2")          #
    .setMaster("spark://10.43.100.121:7077")  # 
)
configuraSgg.setAppName("hpcsparkSgg_preprocesamiento_cluster")

sparkSgg = SparkSession.builder.config(conf=configuraSgg).getOrCreate()
sqlContext = SQLContext(sparkContext=sparkSgg.sparkContext,
                        sparkSession=sparkSgg)

print("MASTER ACTUAL:", sparkSgg.sparkContext.master)

sparkSgg



Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


MASTER ACTUAL: spark://10.43.100.121:7077


In [2]:
supervised_path = "data/supervised/breast_cancer_sgg.csv"
unsupervised_path = "data/unsupervised/iris_sgg.csv"

df_sup_raw = sparkSgg.read.csv(supervised_path, header=True, inferSchema=True)
df_unsup_raw = sparkSgg.read.csv(unsupervised_path, header=True, inferSchema=True)

print("=== Schema supervised ===")
df_sup_raw.printSchema()
print("=== Schema unsupervised ===")
df_unsup_raw.printSchema()

df_sup_raw.show(5)
df_unsup_raw.show(5)


25/11/26 16:41:42 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


=== Schema supervised ===
root
 |-- mean radius: double (nullable = true)
 |-- mean texture: double (nullable = true)
 |-- mean perimeter: double (nullable = true)
 |-- mean area: double (nullable = true)
 |-- mean smoothness: double (nullable = true)
 |-- mean compactness: double (nullable = true)
 |-- mean concavity: double (nullable = true)
 |-- mean concave points: double (nullable = true)
 |-- mean symmetry: double (nullable = true)
 |-- mean fractal dimension: double (nullable = true)
 |-- radius error: double (nullable = true)
 |-- texture error: double (nullable = true)
 |-- perimeter error: double (nullable = true)
 |-- area error: double (nullable = true)
 |-- smoothness error: double (nullable = true)
 |-- compactness error: double (nullable = true)
 |-- concavity error: double (nullable = true)
 |-- concave points error: double (nullable = true)
 |-- symmetry error: double (nullable = true)
 |-- fractal dimension error: double (nullable = true)
 |-- worst radius: double (nu

In [3]:
from pyspark.sql.functions import col, count, isnan, when

print("Supervised rows:", df_sup_raw.count(), "cols:", len(df_sup_raw.columns))
print("Unsupervised rows:", df_unsup_raw.count(), "cols:", len(df_unsup_raw.columns))

df_sup_raw.describe().show()
df_unsup_raw.describe().show()

def nulls_by_column(df, name):
    print(f"Valores nulos por columna en {name}:")
    df.select([
        count(when(col(c).isNull() | isnan(col(c)), c)).alias(c)
        for c in df.columns
    ]).show()

nulls_by_column(df_sup_raw, "supervised")
nulls_by_column(df_unsup_raw, "unsupervised")


Supervised rows: 569 cols: 31
Unsupervised rows: 150 cols: 6


+-------+------------------+-----------------+-----------------+-----------------+--------------------+-------------------+-------------------+--------------------+--------------------+----------------------+------------------+------------------+------------------+-----------------+--------------------+--------------------+--------------------+--------------------+--------------------+-----------------------+------------------+------------------+------------------+-----------------+--------------------+-------------------+-------------------+--------------------+-------------------+-----------------------+------------------+
|summary|       mean radius|     mean texture|   mean perimeter|        mean area|     mean smoothness|   mean compactness|     mean concavity| mean concave points|       mean symmetry|mean fractal dimension|      radius error|     texture error|   perimeter error|       area error|    smoothness error|   compactness error|     concavity error|concave points error|

In [4]:
df_sup = df_sup_raw.dropDuplicates()
df_unsup = df_unsup_raw.dropDuplicates()

numeric_cols_sup = [c for (c, t) in df_sup.dtypes if t in ("int", "double", "float", "bigint")]
numeric_cols_unsup = [c for (c, t) in df_unsup.dtypes if t in ("int", "double", "float", "bigint")]

df_sup = df_sup.na.fill(0, subset=numeric_cols_sup)
df_unsup = df_unsup.na.fill(0, subset=numeric_cols_unsup)

df_sup.printSchema()
df_unsup.printSchema()


root
 |-- mean radius: double (nullable = false)
 |-- mean texture: double (nullable = false)
 |-- mean perimeter: double (nullable = false)
 |-- mean area: double (nullable = false)
 |-- mean smoothness: double (nullable = false)
 |-- mean compactness: double (nullable = false)
 |-- mean concavity: double (nullable = false)
 |-- mean concave points: double (nullable = false)
 |-- mean symmetry: double (nullable = false)
 |-- mean fractal dimension: double (nullable = false)
 |-- radius error: double (nullable = false)
 |-- texture error: double (nullable = false)
 |-- perimeter error: double (nullable = false)
 |-- area error: double (nullable = false)
 |-- smoothness error: double (nullable = false)
 |-- compactness error: double (nullable = false)
 |-- concavity error: double (nullable = false)
 |-- concave points error: double (nullable = false)
 |-- symmetry error: double (nullable = false)
 |-- fractal dimension error: double (nullable = false)
 |-- worst radius: double (nullable

In [5]:
from pyspark.ml.feature import VectorAssembler, StandardScaler

# ---------- SUPERVISADO ----------
label_col = "label"

feature_cols_sup = [c for c in numeric_cols_sup if c != label_col]

assembler_sup = VectorAssembler(
    inputCols=feature_cols_sup,
    outputCol="features_raw"
)
df_sup_vec = assembler_sup.transform(df_sup)

scaler_sup = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)
scaler_model_sup = scaler_sup.fit(df_sup_vec)
df_sup_final = scaler_model_sup.transform(df_sup_vec).select(label_col, "features")

df_sup_final.show(5)

# ---------- NO SUPERVISADO ----------
feature_cols_unsup = [c for c in numeric_cols_unsup if c not in ("target",)]
assembler_unsup = VectorAssembler(
    inputCols=feature_cols_unsup,
    outputCol="features_raw"
)
df_unsup_vec = assembler_unsup.transform(df_unsup)

scaler_unsup = StandardScaler(
    inputCol="features_raw",
    outputCol="features",
    withMean=True,
    withStd=True
)
scaler_model_unsup = scaler_unsup.fit(df_unsup_vec)
df_unsup_final = scaler_model_unsup.transform(df_unsup_vec).select("features")

df_unsup_final.show(5)


+-----+--------------------+
|label|            features|
+-----+--------------------+
|    0|[1.09609952943171...|
|    1|[-0.6206757760066...|
|    1|[-0.4731182290929...|
|    0|[0.48884347097913...|
|    0|[1.28338410820681...|
+-----+--------------------+
only showing top 5 rows

+--------------------+
|            features|
+--------------------+
|[-1.0153732795881...|
|[-0.8950147921906...|
|[0.66964554397658...|
|[-0.5339393299982...|
|[-0.2932223552032...|
+--------------------+
only showing top 5 rows



In [6]:
df_sup_final.write.mode("overwrite").parquet("data/supervised/preprocessed_sgg")
df_unsup_final.write.mode("overwrite").parquet("data/unsupervised/preprocessed_sgg")

print("Datos preprocesados guardados en Parquet.")


Datos preprocesados guardados en Parquet.
